# Portfolio Notebook 1 — Modern Portfolio Theory: Foundations

**Pure theory and computation. No engine. No backtest.**

This notebook is a textbook with runnable cells.
Every concept gets its equation, a plain-English explanation, geometric intuition,
and the code that computes it.
Run every cell offline against `data/egx/`.

---

## What is Modern Portfolio Theory?

In 1952, Harry Markowitz published *Portfolio Selection* in *The Journal of Finance* —
a short paper that gave diversification a rigorous mathematical foundation.
Before Markowitz, "don't put all your eggs in one basket" was folklore.
After Markowitz, it had an equation.

**The problem MPT solves.**
Given a universe of risky assets, how do you allocate capital across them to achieve
the highest possible expected return for a given level of risk — or equivalently,
the lowest possible risk for a given target return?

**The central insight.**
Risk is not just the average riskiness of individual assets.
It depends on how those assets *move together*.
Two volatile assets that tend to rise and fall at *different* times combine into a
portfolio that is less volatile than either asset alone.
Correlation is the mechanism; the efficient frontier is the result.

**Who was Markowitz?**
Harry Markowitz (1927–2023) received the Nobel Memorial Prize in Economic Sciences
in 1990. His 1952 paper introduced the mean–variance framework that underlies
virtually every quantitative portfolio construction method used today — from passive
index funds to institutional factor models.

**The key assumptions:**

1. Returns are fully characterised by their mean and covariance.
   (Equivalently: they are approximately normally distributed, or investors are
   mean–variance rational and care only about the first two moments.)
2. Investors prefer higher expected return and lower variance.
3. No transaction costs, no taxes; markets are liquid enough to achieve any weight.
4. **The historical mean and covariance matrix are the true expected values.**

Assumption 4 is the most dangerous one.
This notebook violates it *deliberately* — we compute $\mu$ and $\Sigma$ from the
*full sample*, which includes data that would not have existed at the start of the
history. Every number that follows is therefore an in-sample fiction.

**Notebook 2 demonstrates exactly how dishonest that is in practice.**


In [ ]:
%matplotlib inline
import sys, os, pathlib
import warnings

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize

warnings.filterwarnings("ignore", category=RuntimeWarning)

# Colour palette — mirrors charts.py so every chart looks like one lab
_BG      = "#0d1117"
_BG_AXES = "#161b22"
_GRID    = "#21262d"
_TEXT    = "#c9d1d9"
_BORDER  = "#30363d"
_BLUE    = "#58a6ff"
_ORANGE  = "#f0883e"
_GREEN   = "#3fb950"
_RED     = "#f85149"
_YELLOW  = "#d29922"
_PURPLE  = "#bc8cff"


## 1. Load Data

We load every CSV in `data/egx/` whose symbol is not the market index (`EGX30`) and
which has at least one full calendar year of history (≥ 252 trading-day bars).
This threshold keeps only investable equities with enough history to estimate a
covariance matrix.

The table below immediately exposes the **asymmetric-history problem**: FWRY was
listed significantly later than the other stocks. Because a covariance matrix
requires *simultaneous* observations across all assets, we can only use dates where
every symbol has a return. The common sample starts at FWRY's first observation.
Earlier data for the longer-running stocks is discarded.

This information loss is one of many estimation-error sources in MPT.
Notebook 3 (walk-forward) shows how to handle it without discarding history.


In [ ]:
from tradinglab.data_feed import DataFeed


feed = DataFeed.from_dir("data/egx")  # aligns every symbol onto the shared trading calendar
symbols = feed.symbols
n = feed.n_assets

print(f"Universe: {n} symbols\n")


### Building the returns matrix

The raw material for MPT is the *return* of each asset on each day, not the
price level. We use the simple (arithmetic) one-day return:

$$r_{i,t} = \frac{P_{i,t} - P_{i,t-1}}{P_{i,t-1}}$$

where $P_{i,t}$ is the adjusted closing price of asset $i$ on day $t$.

We then align all series on the **intersection** of trading dates — rows where
every symbol has an observation — and drop the first row per series (which is
`NaN` after `pct_change()`).

**Annualisation factor.** The metrics layer in this lab derives the annualisation
factor from observed bar counts, not from a calendar constant (STRUCTURE.md §6).
Egypt's trading calendar shifts every year due to moving Islamic holidays and ad-hoc
EGX closures. We compute:

$$\text{ann\_factor} = \frac{\text{observed bars}}{\text{calendar years spanned}}$$

and report the value used so short samples are never presented as full years.


In [ ]:
# feed.returns row 0 is a placeholder 0.0 (no prior day to compare against) — drop it.
# NOTE: full-sample is for TEACHING the shapes — section 4 explains why it's
# actually cheating, and notebook 2 (naive deployment) fixes it.
R = feed.returns[1:]
returns = pd.DataFrame(R, index=feed.dates[1:], columns=symbols)

print(f"Common sample: {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"  {len(returns):,} trading days across {n} symbols")

# Annualisation factor: observed bars ÷ calendar years
_cal_years = (feed.dates[-1] - feed.dates[0]).days / 365.25
ann = len(returns) / _cal_years
print(f"\nAnnualisation factor: {ann:.2f}  "
      f"({len(returns):,} bars / {_cal_years:.3f} calendar years)")


## 1.5 — Four Words You Need Before Any of the Math

MPT is built out of four measurements. They sound similar and get mixed up
constantly, so before any matrices appear, here is exactly what each one is.

---

### Variance — how much does *one* thing bounce around?

$$\text{Var}(r) = \text{average of } (r - \bar{r})^2$$

Take each return, subtract the average return, square it, then average those
squared differences. Big swings away from average produce a big number.

**The catch:** squaring means the answer is in *squared* units. If returns are
in percent, variance is in percent-squared — a number like `0.0004` that nobody
can look at and intuitively judge.

---

### Volatility — the same thing, in units you can actually read

$$\sigma = \sqrt{\text{Var}(r)}$$

Just the square root of variance, which puts it back into the same units as
the returns themselves. "This stock moves about 2% a day" is something you can
reason about. "This stock has variance 0.0004" is not.

**So which one measures risk?** They measure the *same underlying thing* —
how spread out the returns are. But **volatility is the number people report
and talk about**, precisely because the square root makes it readable. Variance
is what appears *inside* the equations, and gets square-rooted at the end to
produce the risk figure a human actually looks at.

*You have already been computing this since week 1 — the `volatility` feature
is a rolling standard deviation of returns.*

---

### Covariance — do *two* things move together?

$$\text{Cov}(r_A, r_B) = \text{average of } (r_A - \bar{r}_A)(r_B - \bar{r}_B)$$

Instead of "how much does this one bounce around its own average," covariance
asks: when A is above its average, does B also tend to be above its own?

- **Positive** — they tend to rise and fall together.
- **Negative** — one tends to rise when the other falls.
- **Near zero** — no consistent relationship.

**One fact that ties it all together:** variance is just covariance of something
with itself.

$$\text{Var}(r) = \text{Cov}(r, r)$$

They are not two separate ideas. Variance is the special case where both assets
are the same asset.

---

### Correlation — covariance, rescaled so you can compare

Covariance is awkward to read on its own, because its size depends on how
volatile each asset already was. A covariance of `0.001` could mean two calm
assets moving closely together, *or* two wild assets barely related — you
cannot tell which. Correlation fixes that:

$$\rho_{AB} = \frac{\text{Cov}(r_A, r_B)}{\sigma_A \times \sigma_B}$$

Dividing by both volatilities forces the answer into a fixed, readable range:

| $\rho$ | Meaning |
|--------|---------|
| $+1$   | move perfectly together |
| $0$    | no linear relationship |
| $-1$   | move perfectly opposite |

**This is what people mean when they casually say two stocks are "correlated."**

In [ ]:
# All four, computed on two real stocks from our universe.
a, b = symbols[0], symbols[1]
ra, rb = returns[a], returns[b]

var_a  = ra.var()
vol_a  = ra.std()
cov_ab = returns[[a, b]].cov().loc[a, b]
corr_ab = returns[[a, b]].corr().loc[a, b]

print(f"Using {a} and {b}, on DAILY returns:\n")
print(f"  Variance of {a}:     {var_a:.6f}   <- squared units, hard to read")
print(f"  Volatility of {a}:   {vol_a:.6f}   <- same number, square-rooted = {vol_a*100:.2f}% per day")
print(f"  Cov({a}, {b}):  {cov_ab:.6f}   <- sign matters, size is hard to judge")
print(f"  Corr({a}, {b}): {corr_ab:+.4f}      <- same relationship, rescaled to [-1, +1]")
print()
print(f"  check -- variance IS covariance with itself: {np.isclose(var_a, returns[[a,a]].cov().iloc[0,0])}")
print(f"  check -- corr = cov / (vol_a * vol_b):       {np.isclose(corr_ab, cov_ab / (ra.std() * rb.std()))}")

### Why any of this matters: diversification, with real numbers

Here is the whole argument for MPT in one table. Take **two stocks that are
equally risky** — both 20% volatility — and split your money 50/50 between
them. The *only* thing that changes below is how correlated they are:

Run the cell and read the last column. Nothing about either stock changed —
both are still 20% volatile on their own. Only the *relationship* between them
changed, and the portfolio risk moved from 20% all the way down to 0%.

**That is the entire mechanism.** When two assets are not perfectly correlated,
their bad days do not always land on the same day — one dips while the other
holds, and the swings partially cancel out in the combination. Diversification
is not folk wisdom about eggs and baskets; it is a provable reduction in
variance, and the exact size of that reduction is set by correlation.

In [ ]:
# Two stocks, both 20% volatility, 50/50 split. Only correlation changes.
s1 = s2 = 0.20
w1 = w2 = 0.5

print("Both stocks are 20% volatile. Portfolio is 50/50. Only correlation differs:\n")
print(f"  {'correlation':>12s}   {'portfolio volatility':>20s}   note")
print(f"  {'-'*12}   {'-'*20}   {'-'*30}")

_notes = {
    1.0:  "no benefit at all",
    0.5:  "some benefit",
    0.0:  "lower than EITHER stock alone",
    -0.5: "half the risk of either stock",
    -1.0: "perfect hedge (theoretical only)",
}
for rho in [1.0, 0.5, 0.0, -0.5, -1.0]:
    var_p = w1**2 * s1**2 + w2**2 * s2**2 + 2*w1*w2*rho*s1*s2
    vol_p = np.sqrt(max(var_p, 0.0))
    print(f"  {rho:+12.1f}   {vol_p*100:19.2f}%   {_notes[rho]}")

print("\nNotice: at zero correlation the portfolio is LESS risky than either")
print("stock individually -- 14.14% vs 20% -- without giving up any expected return.")

## 2. Expected Portfolio Return

**The plain version first.** Your portfolio's expected return is just the
weighted average of each stock's expected return — multiply each one by how
much of your money is in it, then add them all up:

$$\mathbb{E}[r_p] = w_1 \mu_1 + w_2 \mu_2 + \dots + w_n \mu_n = \sum_i w_i \mu_i$$

where $w_i$ is the fraction of your capital in asset $i$ (all the $w_i$ add up
to 1), and $\mu_i$ is that asset's expected return.

**The same thing, written compactly.** That sum is exactly what a dot product
does, so it is usually written as one short expression — this is the form you
will see in every textbook, and the form the code uses:

$$\boxed{\mathbb{E}[r_p] = w^\top \mu}$$

These two equations say the identical thing. The second is just shorthand.

**Plain English.**
If you put 40% in an asset returning 20%/yr and 60% in one returning 10%/yr,
the portfolio is expected to return $0.4 \times 0.20 + 0.6 \times 0.10 = 14\%$/yr.
There is nothing non-trivial here — expected return is linear in the weights.

**Intuition.**
The interesting result is *not* about expected returns but about variance (see §3).
Two portfolios can have exactly the same expected return but very different risk.
MPT chooses the one with lower risk.

We estimate $\mu$ as the annualised sample mean of each asset's daily returns:

$$\hat{\mu}_i = \bar{r}_i \times \text{ann\_factor}$$

where $\bar{r}_i = \frac{1}{T}\sum_{t=1}^{T} r_{i,t}$.


In [ ]:
# mu: annualised mean return for each asset.
# Variable name matches the math throughout this notebook.
mu = returns.mean() * ann  # pd.Series, index = symbols

# Bar chart for visual comparison
fig, ax = plt.subplots(figsize=(8, 4))
colors = [_GREEN if v >= 0 else _RED for v in mu.values]
ax.bar(symbols, mu.values * 100, color=colors, width=0.6, zorder=2)
ax.axhline(0, color=_BORDER, linewidth=0.8)
ax.set_ylabel("Annualised mean return (%)")
ax.set_title(r"$\mu$ — Expected annual return per symbol", pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## 3. Portfolio Variance

The expected return formula is unremarkable — it is just a weighted average.
What Markowitz showed is that *variance* has a very different structure.

**Start with the two-stock case, where you can see every term.** For just two
assets, portfolio variance is:

$$\sigma_p^2 = \underbrace{w_1^2 \sigma_1^2 + w_2^2 \sigma_2^2}_{\text{each stock's own risk}} + \underbrace{2 w_1 w_2 \, \text{Cov}(r_1, r_2)}_{\text{how they move together}}$$

**This is exactly the formula you already ran in §1.5** — the one that took
two 20%-volatility stocks down to 14.14% at zero correlation. The first two
terms are each stock's own risk. The third term is the one that does all the
interesting work: if the two stocks move *oppositely*, the covariance is
negative, and that term actively *subtracts* from total risk.

**Now the general case.** With $n$ assets you need every pair, so the single
cross-term becomes a double sum over all $i$ and $j$:

$$\sigma_p^2 = \sum_i \sum_j w_i \, \text{Cov}(r_i, r_j) \, w_j$$

Collect all those pairwise covariances into an $n \times n$ grid — the
**covariance matrix** $\Sigma$, where $\Sigma_{ij} = \text{Cov}(r_i, r_j)$ —
and the whole double sum compresses into one expression:

$$\boxed{\sigma_p^2 = w^\top \Sigma\, w}$$

All three equations above are the same statement, written at three levels of
compactness. If the matrix form looks opaque, mentally expand it back to the
two-stock version — that is all it is doing, just for every pair at once.

**Plain English.**
Expand the double sum. The diagonal terms $w_i^2 \Sigma_{ii} = w_i^2 \sigma_i^2$
are the individual asset variances, weighted by the square of the allocation.
The off-diagonal terms $2 w_i w_j \Sigma_{ij} = 2 w_i w_j \rho_{ij} \sigma_i \sigma_j$
are the covariance contributions, where $\rho_{ij}$ is the correlation between
assets $i$ and $j$.

**The central insight.**
If two assets have high positive correlation ($\rho_{ij} \approx 1$), combining
them adds nearly the full variance of each. If they are uncorrelated
($\rho_{ij} \approx 0$), the cross-terms vanish and the portfolio variance is
lower than the weighted average of individual variances. If they are negatively
correlated ($\rho_{ij} < 0$), the cross-terms are *negative* — they actively
reduce portfolio variance.

This is why diversification has a mathematical payoff: you can lower risk *without*
sacrificing expected return, as long as you pick assets that do not move in lockstep.

We estimate $\Sigma$ as the annualised sample covariance matrix:

$$\hat{\Sigma} = \text{Cov}(R) \times \text{ann\_factor}$$

where $\text{Cov}(R)$ is the $n \times n$ matrix of pairwise sample covariances
of daily returns.


In [ ]:
# Sigma: annualised covariance matrix.
# Variable name matches the math throughout this notebook.
Sigma = returns.cov() * ann  # pd.DataFrame, shape (n, n)

# The correlation matrix is easier to read (all values in [-1, 1]).
# We display correlation; the optimiser uses the covariance matrix.
corr = returns.corr()

print("Sigma (annualised covariance matrix):\n")
print(Sigma.to_string(float_format=lambda x: f"{x:.4f}"))

# ── Heatmap ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdYlGn", aspect="auto")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(symbols, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(symbols, fontsize=10)

for i in range(n):
    for j in range(n):
        val = corr.iloc[i, j]
        # Dark text on light cells, light text on dark cells
        txt_color = "#0d1117" if abs(val) < 0.6 else _TEXT
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color=txt_color, fontsize=9, fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Pearson correlation  $\\rho_{ij}$", color=_TEXT)
cbar.ax.yaxis.set_tick_params(color=_TEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=_TEXT)

ax.set_title(
    r"Return Correlation Matrix  ($\rho_{ij}$)" + "\n"
    "Green = low correlation → diversification benefit; "
    "Red = high correlation → limited benefit",
    pad=14,
)
plt.tight_layout()
plt.show()

print("\nNote: the optimiser uses Sigma (covariance), not the correlation matrix.")
print("      Correlation is displayed because all values share a common [-1,1] scale.")


## 4. The Sharpe Ratio in Portfolio Form

The Sharpe ratio measures the excess return earned *per unit of total risk*.
In portfolio form:

$$\boxed{\text{Sharpe}(w) = \frac{w^\top \mu - r_f}{\sqrt{w^\top \Sigma\, w}}}$$

where $r_f$ is the annual risk-free rate.

**Plain English.**
The numerator is the annualised portfolio return above the risk-free hurdle.
The denominator is the annualised portfolio volatility ($\sigma_p$).
The ratio asks: "for every one percent of volatility I accept, how much *extra*
return (above T-bills) do I get?"

**Interpretation (from `docs/METRICS.md`):**

| Sharpe        | Verdict                           |
|---------------|-----------------------------------|
| $< 0$         | Lost to risk-free; risk unpaid    |
| $0$ – $1$     | Positive but unremarkable         |
| $1$ – $2$     | Good                              |
| $> 2$         | Excellent — suspicious on backtest|

**Egyptian context.**
The EGX default risk-free rate is the 91-day T-bill yield: approximately **27.25%**
(high by global standards because Egypt runs a high-rate monetary environment).
This sets a tough hurdle — far higher than the US (~4–5%). A strategy must earn
well above 27% annually before the Sharpe numerator even turns positive.

**The tangency portfolio.**
Among all portfolios on the efficient frontier, the one that *maximises* the Sharpe
ratio is called the **tangency portfolio** or **max-Sharpe portfolio**. Geometrically
it is the point where the *capital market line* — the straight line from the risk-free
rate through mean–variance space — is tangent to the efficient frontier curve.


In [ ]:
RF = 0.2725  # Egyptian 91-day T-bill rate (docs/METRICS.md default)

def sharpe(w: np.ndarray) -> float:
    # Annualised Sharpe ratio for weight vector w.
    # Uses module-level mu and Sigma (full sample).
    # Variable names match the math: mu, Sigma, RF.
    w = np.asarray(w, dtype=float)
    port_return = float(w @ mu.values)
    port_vol    = float(np.sqrt(w @ Sigma.values @ w))
    if port_vol < 1e-12:
        return 0.0
    return (port_return - RF) / port_vol

# Sanity check on equal weights
w_check = np.ones(n) / n
print(f"Risk-free rate (rf):        {RF*100:.2f}%/yr")
print(f"Equal-weight return:        {(w_check @ mu.values)*100:.2f}%/yr")
print(f"Equal-weight volatility:    {np.sqrt(w_check @ Sigma.values @ w_check)*100:.2f}%/yr")
print(f"Equal-weight Sharpe:        {sharpe(w_check):.4f}")


## 5. The Efficient Frontier

**Definition.**
A portfolio is *efficient* if there is no other portfolio that has:
- the same expected return with *lower* variance, or
- the same variance with *higher* expected return.

The **efficient frontier** is the locus (set of all) efficient portfolios —
the upper-left boundary of the feasible set in mean–variance space.

**Computing it numerically.**
For each target return $\mu^*$, we solve the **minimum-variance problem**:

$$\min_{w}\; \sigma_p^2 = w^\top \Sigma\, w$$

$$\text{subject to}\quad w^\top \mu = \mu^*,\quad \mathbf{1}^\top w = 1,\quad w \geq 0$$

The last constraint ($w \geq 0$) enforces the *long-only* restriction:
no short-selling. This shrinks the feasible set relative to the unconstrained
Markowitz frontier, but it is realistic for this lab.

We sweep $\mu^*$ from $\min_i \mu_i$ to $\max_i \mu_i$ in small steps,
solve each optimisation with `scipy.optimize.minimize` (SLSQP),
and collect the resulting $(\\sigma_p, \mu^*)$ pairs.

**The minimum-variance portfolio (MVP).**
The leftmost point on the efficient frontier — the portfolio with the lowest
possible variance among *all* long-only portfolios, regardless of return.
No allocation preferences are needed to identify it; it is pure risk minimisation.

**The tangency portfolio (max-Sharpe).**
The point on the frontier that maximises the Sharpe ratio.
It balances return enhancement against risk reduction relative to the T-bill hurdle.


In [ ]:
mu_arr  = mu.values.copy()    # shape (n,)
Sig_arr = Sigma.values.copy() # shape (n, n)


def _port_variance(w: np.ndarray) -> float:
    return float(w @ Sig_arr @ w)


def _min_var_for_target(mu_target: float):
    # Return scipy OptimizeResult for the min-variance problem at mu_target.
    constraints = [
        {"type": "eq", "fun": lambda w: float(w.sum()) - 1.0},           # sum = 1
        {"type": "eq", "fun": lambda w: float(w @ mu_arr) - mu_target},  # return target
    ]
    bounds = [(0.0, 1.0)] * n   # long-only
    x0 = np.ones(n) / n
    return minimize(
        _port_variance, x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 2000},
    )


# Sweep target returns
mu_lo = float(mu_arr.min())
mu_hi = float(mu_arr.max())
target_returns = np.linspace(mu_lo, mu_hi, 300)

frontier_vols: list[float] = []
frontier_rets: list[float] = []
frontier_wts:  list[np.ndarray] = []

for mu_t in target_returns:
    res = _min_var_for_target(mu_t)
    if res.success:
        frontier_vols.append(float(np.sqrt(res.fun)))
        frontier_rets.append(mu_t)
        frontier_wts.append(res.x.copy())

frontier_vols = np.array(frontier_vols)
frontier_rets = np.array(frontier_rets)
frontier_wts  = np.array(frontier_wts)

print(f"Frontier computed: {len(frontier_vols)} points "
      f"(return range {mu_lo*100:.1f}% – {mu_hi*100:.1f}%)")


In [ ]:
# ── Minimum-Variance Portfolio (MVP) ──────────────────────────────────────────
idx_mvp = int(np.argmin(frontier_vols))
mvp_vol = frontier_vols[idx_mvp]
mvp_ret = frontier_rets[idx_mvp]
mvp_wts = frontier_wts[idx_mvp]
mvp_sharpe = (mvp_ret - RF) / mvp_vol

# ── Max-Sharpe (Tangency) Portfolio ─────────────────────────────────────────
frontier_sharpes = (frontier_rets - RF) / frontier_vols
idx_msp = int(np.argmax(frontier_sharpes))
msp_vol = frontier_vols[idx_msp]
msp_ret = frontier_rets[idx_msp]
msp_wts = frontier_wts[idx_msp]
msp_sharpe = frontier_sharpes[idx_msp]

print("Minimum-Variance Portfolio (MVP)")
print(f"  Return  : {mvp_ret*100:.2f}%/yr")
print(f"  Volatility: {mvp_vol*100:.2f}%/yr")
print(f"  Sharpe  : {mvp_sharpe:.4f}")
print(f"  Weights : {dict(zip(symbols, mvp_wts.round(4)))}\n")

print("Max-Sharpe (Tangency) Portfolio")
print(f"  Return  : {msp_ret*100:.2f}%/yr")
print(f"  Volatility: {msp_vol*100:.2f}%/yr")
print(f"  Sharpe  : {msp_sharpe:.4f}")
print(f"  Weights : {dict(zip(symbols, msp_wts.round(4)))}")


## 6. Equal-Weight Baseline

Before we can claim any benefit from optimisation, we need a baseline.
The natural one is the **1/N portfolio**: allocate equal weight $w_i = 1/n$ to
every asset in the universe.

Equal-weight has an honourable track record. DeMiguel, Garlappi, and Uppal (2009)
showed empirically that $1/N$ beats most optimised strategies out-of-sample,
because estimation error in $\mu$ and $\Sigma$ often more than offsets the
theoretical gain from optimisation.

**This is the bar to beat.**
An optimiser that does not outperform equal-weight on a forward-looking basis
has accomplished nothing — it has only fit the history harder.
The closing reflection below explains why this section's optimised portfolios
should not be taken at face value.


In [ ]:
w_ew   = np.ones(n) / n           # w: equal weight vector
ew_ret = float(w_ew @ mu_arr)     # E[r_p] = w^T mu
ew_vol = float(np.sqrt(w_ew @ Sig_arr @ w_ew))  # sigma_p = sqrt(w^T Sigma w)
ew_sharpe = (ew_ret - RF) / ew_vol

print("Equal-Weight Portfolio (1/N)")
print(f"  Weight per asset : {1/n*100:.1f}%")
print(f"  Return           : {ew_ret*100:.2f}%/yr")
print(f"  Volatility       : {ew_vol*100:.2f}%/yr")
print(f"  Sharpe           : {ew_sharpe:.4f}")
print()
print("Context relative to frontier portfolios:")
print(f"  MVP Sharpe       : {mvp_sharpe:.4f}")
print(f"  Max-Sharpe       : {msp_sharpe:.4f}")
print(f"  Equal-Wt Sharpe  : {ew_sharpe:.4f}")


In [ ]:
# Per-asset volatility and return — used to plot each stock on the frontier chart.
stock_vols = np.sqrt(np.diag(Sig_arr))   # annualised vol per symbol
stock_rets = mu_arr.copy()               # annualised return per symbol


### Mean–Variance Chart

The chart below places everything in the same $(\\sigma_p, \mathbb{E}[r_p])$ space.

- **Blue curve** — the efficient frontier (minimum variance for each target return).
- **Dots** — individual stocks. Notice they all lie to the *right* of the frontier:
  every single asset, held alone, is dominated — there exists a portfolio with the
  same expected return but lower risk, or higher return at the same risk.
  This is the geometric proof of diversification.
- **Green star** — minimum-variance portfolio (leftmost point on the frontier).
- **Orange star** — max-Sharpe (tangency) portfolio.
- **Red diamond** — equal-weight portfolio ($1/N$).
- **Dashed line** — the risk-free rate, plotted at zero volatility.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# ── Efficient frontier curve ─────────────────────────────────────────────────
ax.plot(
    frontier_vols * 100, frontier_rets * 100,
    color=_BLUE, linewidth=2.5, label="Efficient frontier", zorder=3,
)

# ── Individual stocks ────────────────────────────────────────────────────────
_stock_colors = [_PURPLE, _ORANGE, "#79c0ff", "#a5d6ff", "#56d364", "#ffa657", "#ff7b72"]
for i, sym in enumerate(symbols):
    col = _stock_colors[i % len(_stock_colors)]
    ax.scatter(
        stock_vols[i] * 100, stock_rets[i] * 100,
        s=90, color=col, zorder=5, label=sym,
    )
    ax.annotate(
        sym,
        (stock_vols[i] * 100, stock_rets[i] * 100),
        textcoords="offset points", xytext=(7, 2),
        color=col, fontsize=9, fontweight="bold",
    )

# ── MVP marker ───────────────────────────────────────────────────────────────
ax.scatter(
    mvp_vol * 100, mvp_ret * 100,
    s=220, color=_GREEN, marker="*", zorder=6,
    label=f"Min-variance (Sharpe = {mvp_sharpe:.2f})",
)

# ── Max-Sharpe (tangency) marker ─────────────────────────────────────────────
ax.scatter(
    msp_vol * 100, msp_ret * 100,
    s=220, color=_ORANGE, marker="*", zorder=6,
    label=f"Max-Sharpe / tangency (Sharpe = {msp_sharpe:.2f})",
)

# ── Equal-weight marker ──────────────────────────────────────────────────────
ax.scatter(
    ew_vol * 100, ew_ret * 100,
    s=160, color=_RED, marker="D", zorder=6,
    label=f"Equal-weight 1/N (Sharpe = {ew_sharpe:.2f})",
)

# ── Risk-free rate reference line ────────────────────────────────────────────
ax.axhline(
    y=RF * 100, color=_YELLOW, linestyle="--", linewidth=1.2, alpha=0.7,
    label=f"Risk-free rate  rf = {RF*100:.2f}%",
)

# ── Labels & formatting ──────────────────────────────────────────────────────
ax.set_xlabel("Annualised Volatility  $\\sigma_p$ (%)", fontsize=11)
ax.set_ylabel("Annualised Expected Return  $\\mathbb{E}[r_p]$ (%)", fontsize=11)
ax.set_title(
    "Efficient Frontier — EGX Universe\n"
    "(⚠ full-sample, in-sample computation — see closing reflection)",
    pad=14,
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend(loc="lower right", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()


In [ ]:
# Weight breakdown for the two key portfolios — no hardcoded values.
import pandas as pd

_portfolios = {
    "Min-Variance (MVP)":  mvp_wts,
    "Max-Sharpe":          msp_wts,
    "Equal-Weight (1/N)":  w_ew,
}

wt_df = pd.DataFrame(_portfolios, index=symbols) * 100
wt_df.index.name = "Symbol"
wt_df.columns.name = "Portfolio"

print("Weight allocations (%):\n")
print(wt_df.to_string(float_format=lambda x: f"{x:.2f}%"))


## Closing Reflection — The Look-Ahead Trap

Every number produced in this notebook used the **full sample**.

- $\mu$ was estimated from returns that extend to the last bar in the dataset.
- $\Sigma$ was estimated from the same full-sample covariance.
- The "max-Sharpe portfolio" and "efficient frontier" were computed using data
  that would not have existed at the *start* of the history.

In plain terms: the portfolio labelled "max-Sharpe" was optimised with knowledge
of the future. It is not a strategy — it is a post-hoc rationalisation.
Any investor who tried to run it in real time would have been computing $\mu$
and $\Sigma$ from the data available *up to that moment*, which looks nothing
like the full-sample estimates.

This is the **look-ahead trap** in portfolio construction:
the attractive in-sample numbers are not attainable in practice, because the
information used to compute them did not exist when the decisions had to be made.

**Notebook 2** (*Naive Deployment*) demonstrates exactly how dishonest this is.
It deploys the same portfolios in a walk-forward simulation — where at each
rebalance date the strategy only sees history up to that date — and shows the
gap between the in-sample illusion presented here and the out-of-sample reality.
The gap is always large, and understanding *why* it is large is the central lesson
of the portfolio series.

---

## One More Reason to Distrust That Chart: The Error Maximizer

Look-ahead is not the only problem. There is a second, subtler one, and it is
visible right there in the weight table above.

**Notice how concentrated the optimised portfolios are.** The minimum-variance
portfolio put **100% into a single stock**. The max-Sharpe portfolio spread
across only about seven names out of thirty-four, leaving the other
twenty-seven at exactly zero. An optimiser handed thirty-four assets chose to
ignore most of them entirely.

That is not the optimiser malfunctioning. It is doing precisely what it was
asked — and that is the problem.

**Why it happens.** MPT needs two inputs: expected returns ($\mu$) and the
covariance matrix ($\Sigma$). Covariance is comparatively stable and
estimable. **Expected returns are not.** A stock's true expected return is
extremely hard to estimate from historical data — the noise is large relative
to the signal, and a few lucky months can lift an estimate substantially.

Now consider what the optimiser does with that. It has no way to know which
estimates are reliable and which are noise. It simply finds whichever weights
maximise the objective — so it piles into whatever asset *happened* to have
the most flattering estimate. If a stock's return was overestimated by chance,
the optimiser does not hedge against that possibility; it leans in harder.

**This is why mean-variance optimisers are sometimes called
"error maximizers."** They do not merely reflect estimation error — they
actively amplify it, concentrating capital exactly where the input data was
most optimistically wrong.

**Which is also why the humble equal-weight (1/N) baseline is such a serious
competitor**, and why it appears in this notebook at all. It estimates nothing,
so it cannot amplify estimation error. Notebook 2 puts both to the honest
test — out of sample, where estimates have to be made before the outcome is
known.